In [1]:
# /// script
# requires-python = ">=3.12"
# dependencies = [
#     "anndata>=0.12.11",
#     "scanpy>=1.12.1",
#     "zarr>=3.1.6",
# ]
# ///

In [1]:
from pathlib import Path

In [2]:
import anndata as ad
import numpy as np
import scanpy as sc

In [3]:
INPUT_PATH = Path("habib17.h5ad")
RAW_OUTPUT_PATH = Path("test-data/habib17.zarr")
OUTPUT_PATH = Path("test-data/habib17-differential-expression-test-data.zarr")
GROUPBY_COLUMN = "CellType"

In [4]:
def _sample_expression_values(x: object, max_items: int = 10000) -> np.ndarray:
    if hasattr(x, "tocoo"):
        values = np.asarray(x.data)
    else:
        values = np.asarray(x).ravel()

    if values.size == 0:
        return values

    if values.size > max_items:
        step = max(1, values.size // max_items)
        values = values[::step]

    return values

In [5]:
def _needs_preprocessing(adata: ad.AnnData) -> tuple[bool, str]:
    if "log1p" in adata.uns:
        return False, "Detected adata.uns['log1p']; matrix appears already log-transformed."

    values = _sample_expression_values(adata.X)
    if values.size == 0:
        return True, "Empty matrix sample; applying preprocessing by default."

    tol = 1e-6
    non_integer_fraction = float(np.mean(np.abs(values - np.round(values)) > tol))
    max_value = float(np.max(values))

    # Heuristic: mostly non-integers with compressed range usually indicates logged data.
    if non_integer_fraction > 0.2 and max_value < 50:
        return (
            False,
            (
                "Expression values look already transformed "
                f"(non-integer fraction={non_integer_fraction:.3f}, max={max_value:.3f})."
            ),
        )

    return (
        True,
        (
            "Expression values look like raw counts "
            f"(non-integer fraction={non_integer_fraction:.3f}, max={max_value:.3f})."
        ),
    )

In [6]:
def main() -> None:
    if not INPUT_PATH.exists():
        raise FileNotFoundError(f"Input file not found: {INPUT_PATH}")

    adata = ad.read_h5ad(INPUT_PATH)

    if GROUPBY_COLUMN not in adata.obs.columns:
        available = ", ".join(adata.obs.columns.astype(str).tolist())
        raise KeyError(
            f"Column '{GROUPBY_COLUMN}' not found in obs. Available columns: {available}"
        )

    # Align with the zarr v3 output style used in other dataset creation scripts.
    ad.settings.zarr_write_format = 3
    ad.settings.write_csr_csc_indices_with_min_possible_dtype = True
    ad.settings.auto_shard_zarr_v3 = True

    RAW_OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
    adata.write_zarr(RAW_OUTPUT_PATH)

    print(f"Wrote source AnnData to {RAW_OUTPUT_PATH}")

    # Make sure group labels are categorical for rank_genes_groups.
    adata.obs[GROUPBY_COLUMN] = adata.obs[GROUPBY_COLUMN].astype("category")

    should_preprocess, reason = _needs_preprocessing(adata)
    print(reason)

    if should_preprocess:
        sc.pp.normalize_total(adata, target_sum=1e4)
        sc.pp.log1p(adata)
        print("Applied preprocessing: normalize_total + log1p")
    else:
        print("Skipped preprocessing")

    sc.tl.rank_genes_groups(
        adata,
        groupby=GROUPBY_COLUMN,
        method="wilcoxon",
        corr_method="benjamini-hochberg",
        pts=True,
        key_added="diff01",
        use_raw=False,
    )

    
    sc.tl.rank_genes_groups(
        adata,
        groupby=GROUPBY_COLUMN,
        groups=["ASC2"],
        method="wilcoxon",
        corr_method="benjamini-hochberg",
        pts=True,
        key_added="diff02",
        use_raw=False,
    )

    OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
    adata.write_zarr(OUTPUT_PATH)

    print(f"Wrote DE results to {OUTPUT_PATH}")

In [7]:
if __name__ == "__main__":
    main()

/home/klaus/.cache/uv/environments-v2/juv-tmp-h3rf7-83-b4e02436c468f36f/lib/python3.12/site-packages/zarr/core/array.py:4442: ZarrUserWarning: Automatic shard shape inference is experimental and may change without notice.
  shard_shape_parsed, chunk_shape_parsed = _auto_partition(
/home/klaus/.cache/uv/environments-v2/juv-tmp-h3rf7-83-b4e02436c468f36f/lib/python3.12/site-packages/zarr/api/asynchronous.py:231: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


Wrote source AnnData to test-data/habib17.zarr
Expression values look already transformed (non-integer fraction=1.000, max=5.169).
Skipped preprocessing


/home/klaus/.cache/uv/environments-v2/juv-tmp-h3rf7-83-b4e02436c468f36f/lib/python3.12/site-packages/zarr/core/array.py:4442: ZarrUserWarning: Automatic shard shape inference is experimental and may change without notice.
  shard_shape_parsed, chunk_shape_parsed = _auto_partition(
/home/klaus/.cache/uv/environments-v2/juv-tmp-h3rf7-83-b4e02436c468f36f/lib/python3.12/site-packages/zarr/core/dtype/npy/structured.py:591: UnstableSpecificationWarning: The data type (Struct(fields=(('ASC1', FixedLengthUTF32(length=17, endianness='little')), ('ASC2', FixedLengthUTF32(length=17, endianness='little')), ('END', FixedLengthUTF32(length=17, endianness='little')), ('GABA1', FixedLengthUTF32(length=17, endianness='little')), ('GABA2', FixedLengthUTF32(length=17, endianness='little')), ('MG', FixedLengthUTF32(length=17, endianness='little')), ('NSC', FixedLengthUTF32(length=17, endianness='little')), ('ODC1', FixedLengthUTF32(length=17, endianness='little')), ('OPC', FixedLengthUTF32(length=17, endi

Wrote DE results to test-data/habib17-differential-expression-test-data.zarr


/home/klaus/.cache/uv/environments-v2/juv-tmp-h3rf7-83-b4e02436c468f36f/lib/python3.12/site-packages/zarr/core/dtype/npy/structured.py:591: UnstableSpecificationWarning: The data type (Struct(fields=(('ASC2', FixedLengthUTF32(length=17, endianness='little')),))) does not have a Zarr V3 specification. That means that the representation of arrays saved with this data type may change without warning in a future version of Zarr Python. Arrays stored with this data type may be unreadable by other Zarr libraries. Use this data type at your own risk! Check https://github.com/zarr-developers/zarr-extensions/tree/main/data-types for the status of data type specifications for Zarr V3.
  v3_unstable_dtype_warning(self)
/home/klaus/.cache/uv/environments-v2/juv-tmp-h3rf7-83-b4e02436c468f36f/lib/python3.12/site-packages/zarr/core/dtype/npy/structured.py:591: UnstableSpecificationWarning: The data type (Struct(fields=(('ASC2', Float32(endianness='little')),))) does not have a Zarr V3 specification. 

## Results

In [8]:
adata = ad.read_zarr(OUTPUT_PATH)
adata


AnnData object with n_obs × n_vars = 13067 × 5782
    obs: 'CellType', 'n_counts', 'log1p_n_counts', 'n_genes', 'log1p_n_genes', 'percent_mito', 'percent_ribo', 'percent_hb', 'percent_top50'
    var: 'gene_ids', 'mito', 'ribo', 'hb', 'n_counts', 'n_cells', 'n_genes', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'diff01', 'diff02', 'leiden', 'neighbors', 'pca'
    obsm: 'X_umap'
    varm: 'PCs'
    obsp: 'connectivities', 'distances'

In [9]:
result_set ="diff02"

In [10]:
adata.uns[result_set].keys()

dict_keys(['logfoldchanges', 'names', 'params', 'pts', 'pts_rest', 'pvals', 'pvals_adj', 'scores'])

In [11]:
adata.uns[result_set]["params"]

{'corr_method': 'benjamini-hochberg',
 'groupby': 'CellType',
 'layer': None,
 'method': 'wilcoxon',
 'reference': 'rest',
 'use_raw': False}

## Genes Expressiing

In [12]:
adata.uns[result_set]["pts"]

,ASC2
index,
LINC00115,0.007207
RP11-54O7.1,0.000000
LINC02593,0.012613
SAMD11,0.039640
ISG15,0.045045
...,...
CTA-357J21.1,0.003604
RP11-28F1.2,0.000000
RP11-638I8.1,0.001802


In [13]:
adata.var.head()

,gene_ids,mito,ribo,hb,n_counts,n_cells,n_genes,highly_variable,means,dispersions,dispersions_norm
index,,,,,,,,,,,
LINC00115,21,False,False,False,61.0,65,61,True,0.066755,3.035625,0.803781
RP11-54O7.1,24,False,False,False,25.0,34,25,True,0.023525,2.982569,0.639962
LINC02593,26,False,False,False,23.0,28,21,True,0.029433,3.274476,1.541276
SAMD11,27,False,False,False,50.0,68,45,True,0.055600,3.248546,1.461213
ISG15,34,False,False,False,192.0,243,166,True,0.172761,3.093359,0.982046


## Valeus

In [14]:
adata.uns[result_set]["pvals_adj"]

array([(3.32869752e-202,), (2.60567505e-172,), (1.26441774e-142,), ...,
       (3.23100658e-012,), (1.25016376e-028,), (9.05407999e-052,)],
      shape=(5782,), dtype=[('ASC2', '<f8')])

In [15]:
adata.uns[result_set]["logfoldchanges"]

array([( 5.8159356,), ( 4.308961 ,), ( 5.504424 ,), ..., (-1.6717236,),
       (-3.1463888,), (-3.2507138,)],
      shape=(5782,), dtype=[('ASC2', '<f4')])

In [16]:
adata.uns[result_set]["logfoldchanges"]

array([( 5.8159356,), ( 4.308961 ,), ( 5.504424 ,), ..., (-1.6717236,),
       (-3.1463888,), (-3.2507138,)],
      shape=(5782,), dtype=[('ASC2', '<f4')])

## Genes

In [17]:
adata.uns[result_set]["names"][5781]

np.void(('TTLL7',), dtype=[('ASC2', 'O')])

In [18]:
adata.obs['CellType'].unique().astype(str)

array(['exCA1', 'exCA3', 'ASC1', 'GABA1', 'ODC1', 'exDG', 'Unclassified',
       'exPFC2', 'GABA2', 'END', 'exPFC1', 'MG', 'ASC2', 'OPC', 'NSC'],
      dtype='<U12')

In [19]:
adata.uns[result_set]["names"]['ASC2'][0:5]
for ct in adata.obs['CellType'].unique().astype(str):
    if ct in adata.uns[result_set]["names"].dtype.names:
        print(f"Top 5 Genes of {ct}: {adata.uns[result_set]['names'][ct][0:5]}")

Top 5 Genes of ASC2: ['GFAP' 'CLU' 'AQP4' 'SPARCL1' 'MACF1']


In [20]:
for ct in adata.obs['CellType'].unique().astype(str):
    if ct in adata.uns[result_set]["names"].dtype.names:
        gene = adata.uns[result_set]["names"][ct][0]
        pvalue_adj = adata.uns[result_set]["pvals_adj"][ct][0]
        logfoldchange = adata.uns[result_set]["logfoldchanges"][ct][0]
        pts = adata.uns[result_set]["pts"][ct][0]
        print(f"Top  genes of {ct}: {gene} with adjusted p value of {pvalue_adj}, logfoldchange of {logfoldchange}, and pts of {pts}")


Top  genes of ASC2: GFAP with adjusted p value of 3.3286975205928546e-202, logfoldchange of 5.8159356117248535, and pts of 0.007207207207207207


/tmp/ipykernel_366823/1818202400.py:6: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  pts = adata.uns[result_set]["pts"][ct][0]
